In [1]:
from scipy.stats import linregress
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import colormaps as cmaps
from scipy.interpolate import griddata
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
year = 1980
month = 1
era5_path = f"/work5/ERA5_daily_landgrid_only/global/compressed_sfc_fc_daily_{year}{month:02}.nc4"
ds_era5 = xr.open_dataset(era5_path)

# Read latitude and longitude values from ERA5 data
lat_1d = ds_era5['lat'].values  # shape (141780,)
lon_1d = ds_era5['lon'].values # shape (141780,)

# ============================================================
# 1. read climatology files
# ============================================================
clim_folder = "/work/home/H.extreme/IPCS_extreme/Plot_main_text_Figure/outcomes/"
rad_file = os.path.join(clim_folder, "thirstwave_metrics_growing_season_new.nc")   # Rad. term
tw_feature = xr.open_dataset(rad_file)

intensity = tw_feature["intensity"].values.astype(np.float32)
frequency = tw_feature["event_count"].values.astype(np.float32)
duration = tw_feature["mean_duration"].values 
rad_frac_tw = tw_feature["rad_frac_tw"].values.astype(np.float32)

years = tw_feature["year"].values   
lgrid = tw_feature["lgrid"]

# --------------------------------------------------
# function to compute trend
# --------------------------------------------------
def calc_trend(data, years):
    nyear, ngrid = data.shape
    trend = np.full(ngrid, np.nan, dtype=np.float32)
    pvalue = np.full(ngrid, np.nan, dtype=np.float32)

    for i in range(ngrid):
        y = data[:, i]
        m = np.isfinite(y) & np.isfinite(years)
        if np.sum(m) >= 3:
            res = linregress(years[m], y[m])
            trend[i] = res.slope
            pvalue[i] = res.pvalue

    return trend, pvalue

# --------------------------------------------------
# compute trend
# --------------------------------------------------
int_trend, int_p = calc_trend(intensity, years)
fre_trend, fre_p = calc_trend(frequency, years)
dur_trend, dur_p = calc_trend(duration, years)
rad_frac_trend, rad_frac_p = calc_trend(rad_frac_tw, years)
# --------------------------------------------------
# save nc
# --------------------------------------------------
ds_out = xr.Dataset(
    {
        "intensity_trend": (["lgrid"], int_trend),
        "intensity_pvalue": (["lgrid"], int_p),
        "frequency_trend": (["lgrid"], fre_trend),
        "frequency_pvalue": (["lgrid"], fre_p),
        "duration_trend": (["lgrid"], dur_trend),
        "duration_pvalue": (["lgrid"], dur_p),
        "rad_frac_trend": (["lgrid"], rad_frac_trend),
        "rad_frac_pvalue": (["lgrid"], rad_frac_p),
    },
    coords={"lgrid": lgrid}
)

out_nc = "thirstwave_trend_growing_season.nc"
ds_out.to_netcdf(out_nc)

print("Saved:", out_nc)

Saved: thirstwave_trend_growing_season.nc
